# 04 – Findability Scoring

Apply the rule-based findability score to all receiver candidates.
Inspect score distributions, team-level availability rates and a
missed-opportunity freeze frame.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_competition
from src.features.buildup import filter_buildup
from src.features.line_detection import detect_opponent_lines
from src.features.receiver import detect_receiver_candidates
from src.features.lane import compute_lane_features
from src.features.findability import compute_findability
from src.metrics.outputs import compute_team_metrics, merge_event_scores


In [ ]:
events, frames, lineups, matches = load_competition(competition_id=55, season_id=43)
buildup_df    = filter_buildup(events)
line_df       = detect_opponent_lines(frames, buildup_df)
receivers_df  = detect_receiver_candidates(frames, buildup_df, line_df)
candidates_df = compute_lane_features(receivers_df, frames, buildup_df)
event_scores, candidates_scored = compute_findability(candidates_df)
merged_df     = merge_event_scores(buildup_df, event_scores, line_df)


## Score distribution


In [ ]:
candidates_scored['findability_score'].value_counts().sort_index().plot(
    kind='bar', title='Findability Score Distribution'
)
plt.ylabel('Count')
plt.show()
print(f"\nFindable option available in {merged_df['findable_option_available'].mean()*100:.1f}% of build-up events")


## Team-level availability rate


In [ ]:
team_metrics = compute_team_metrics(merged_df)
display(team_metrics[['team','between_lines_availability_rate','total_buildup_events']].head(20))


## Bar chart – team comparison


In [ ]:
from src.viz.plots import plot_team_comparison_bar
fig = plot_team_comparison_bar(team_metrics)
plt.show()


## Missed opportunity example


In [ ]:
from src.viz.plots import plot_missed_opportunity

findable_ids = event_scores[event_scores['findable_option_available'] == 1]['event_id']
events_with_frames = set(frames['event_id'].unique())
sample_ids = [eid for eid in findable_ids if eid in events_with_frames]

if sample_ids:
    fig = plot_missed_opportunity(
        event_id=sample_ids[0],
        frames_df=frames,
        buildup_df=buildup_df,
        line_df=line_df,
        candidates_scored_df=candidates_scored,
    )
    plt.show()
